<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/w18_verification_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 W18: Kiểm Chứng Kỹ Thuật, Benchmark TACRED & Tự Động Hóa Multi-Seed
### Đề tài: Continual Relation Extraction with Task-Aware Prompt Adaptation (TAPTA)

Notebook này phục vụ các đầu việc kiểm chứng kỹ thuật của **Tuần W18** theo kế hoạch nghiên cứu chính thức (**Track A — Primary Research**):
1. **Kiểm chứng P4 trên Benchmark thứ hai TACRED Track A** (41 quan hệ, 8 tác vụ, strict 5-shot, $M=0$, seed 2021).
2. **Kiểm chứng Bộ công cụ chạy đa hạt giống (Multi-Seed Runner)** (`run_multiseed.py`) chuẩn bị sẵn sàng cho Tuần W19.
3. **Đo đạc chi phí tham số, bộ nhớ tĩnh và độ trễ suy luận trên GPU** (`profile_model_efficiency.py`).
4. *(Tùy chọn - Track B)*: Thăm dò mở rộng dữ liệu Scalability Study nếu GPU dôi dư.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("=" * 60)

## 2. Clone / Cập Nhật Repository `research-code-lab`

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate

## 3. Thực Nghiệm 1: Kiểm Chứng P4 Trên Benchmark Thứ Hai TACRED Track A
*(41 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

In [ ]:
!python experiments/run_p4.py \
    --config configs/p4_hierarchical_prompt_tree_tacred_5shot_seed2021.yaml \
    --device cuda

## 4. Thực Nghiệm 2: Kiểm Thử Bộ Công Cụ Chạy Đa Hạt Giống (Multi-Seed Runner)
*(Chạy 2 hạt giống 2021 & 2022 để kiểm tra pipeline tổng hợp Mean ± Std trước khi sang W19)*

In [ ]:
!python experiments/run_multiseed.py \
    --config configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml \
    --seeds 2021 2022 \
    --device cuda

## 5. Thực Nghiệm 3: Phân Tích Độ Phức Tạp, Tham Số & Bộ Nhớ Tĩnh Trên GPU
*(Chứng minh chỉ tốn 0.0028% tham số/task và ~1.03 MB persistent memory)*

In [ ]:
!python scripts/profile_model_efficiency.py

## 6. (Tùy Chọn - Track B): Thăm Dò Mở Rộng Dữ Liệu (Scalability Study)
*(Chỉ chạy khi GPU còn nhiều quota và đã hoàn tất các mốc Track A ở trên)*

In [ ]:
#@title Lựa chọn thí nghiệm Scalability (Track B) { run: "auto" }
run_scalability = False #@param {type:"boolean"}
scale_setting = "ds10 (10-shot)" #@param ["ds10 (10-shot)", "ds50 (50-shot)", "ds100 (100-shot)", "ds_full (~420-shot)"]

scale_map = {
    "ds10 (10-shot)": "configs/scalability/p4_fewrel_ds10_seed2021.yaml",
    "ds50 (50-shot)": "configs/scalability/p4_fewrel_ds50_seed2021.yaml",
    "ds100 (100-shot)": "configs/scalability/p4_fewrel_ds100_seed2021.yaml",
    "ds_full (~420-shot)": "configs/scalability/p4_fewrel_ds_full_seed2021.yaml",
}

if run_scalability:
    cfg = scale_map[scale_setting]
    print(f"🚀 Đang chạy thí nghiệm Track B: {scale_setting}...")
    !python experiments/run_scalability_study.py --config {cfg} --device cuda --oracle_diagnostic
else:
    print("ℹ️ Bỏ qua Track B để tập trung tối đa tài nguyên cho Track A theo đúng kế hoạch W18.")

## 7. Tổng Hợp Kết Quả & Đóng Gói Tải Về (Download Results Artifact)

In [ ]:
import json
import shutil
from pathlib import Path
from google.colab import files

print("=" * 70)
print("TỔNG HỢP KẾT QUẢ KIỂM CHỨNG TUẦN W18")
print("=" * 70)

# Kiểm tra kết quả TACRED
tacred_sum = Path("runs/p4_hierarchical_tree/tacred_5shot/seed_2021/summary.json")
if tacred_sum.exists():
    with tacred_sum.open() as f:
        s = json.load(f)
    print(f"✅ P4 trên TACRED Track A:")
    print(f"   - Final Decision Fusion Accuracy: {s.get('final_decision_fusion_accuracy', 0)*100:.2f}%")
    print(f"   - Average Incremental Acc (AIA): {s.get('average_incremental_accuracy', 0)*100:.2f}%")
    print(f"   - Whole Forgetting:              {s.get('whole_forgetting', 0)*100:.2f}%")
    print(f"   - Router Top-1 Accuracy:         {s.get('final_router_top1_accuracy', 0)*100:.2f}%")
else:
    print("ℹ️ Chưa có kết quả TACRED. (Hãy chạy Mục 3)")

# Kiểm tra kết quả Multi-Seed
ms_sum = Path("runs/p4_hierarchical_tree/fewrel_5shot/multiseed_summary.json")
if ms_sum.exists():
    with ms_sum.open() as f:
        m = json.load(f)
    print(f"
✅ Multi-Seed Aggregation ({m.get('num_seeds')} seeds):")
    agg = m.get("aggregated_metrics", {})
    for k, v in agg.items():
        print(f"   - {k}: {v.get('formatted')}")
else:
    print("ℹ️ Chưa có kết quả Multi-Seed. (Hãy chạy Mục 4)")

# Đóng gói toàn bộ thư mục runs/
archive_dir = "runs"
zip_name = "/content/w18_verification_results"
if os.path.exists(archive_dir):
    shutil.make_archive(zip_name, "zip", archive_dir)
    print(f"
📦 Đã đóng gói thành công: {zip_name}.zip")
    files.download(f"{zip_name}.zip")
else:
    print("Thư mục runs không tồn tại để đóng gói!")